# 🏫 Proyecto Final: Identificación y Clasificación del Riesgo de Deserción Escolar en Ecuador
**Curso:** Inteligencia Artificial - ESPOL  
**Profesor:** Enrique Peláez J. Ph.D.  
**Grupo #9:** Mateo Mayorga, Anthony Navarrete, Andrés Salinas  
**Dataset:** Ministerio de Educación del Ecuador (MINEDUC) - Datos Abiertos (2009-2024)

---  
### 🎯 Objetivo General
Clasificar el nivel de riesgo de deserción escolar (Bajo, Medio, Alto) en Unidades Educativas ecuatorianas mediante un modelo propio de **Perceptrón Multicapa (MLP)** y comparar su rendimiento frente a 5 modelos de línea base, incorporando explicabilidad con **SHAP**.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

# Añadir el directorio raíz al path
sys.path.append(os.path.abspath('..'))

from src.data_preprocessing import DataPreprocessor
from src.mlp_classifier import MLPClassifier
from src.baseline_evaluator import BaselineEvaluator
from src.shap_explainer import SHAPExplainer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('✅ Librerías e importaciones cargadas con éxito.')

In [ ]:
ruta_inicio = '../data/raw/1Registro-Administrativo-Historico_2009-202X-Inicio.xlsx'
ruta_fin = '../data/raw/2Registro-Administrativo-Historico_2009-2024-Fin.xlsx'

print('🔄 Cargando y procesando pipeline ETL del MINEDUC Ecuador...')
preprocesador = DataPreprocessor(ruta_inicio, ruta_fin)
df_raw = preprocesador.cargar_y_fusionar_datasets(sample_size=4000)
df_clean = preprocesador.limpiar_y_calcular_abandono(df_raw)
df_final = preprocesador.discretizar_riesgo(df_clean)

X, y = preprocesador.transformar_caracteristicas(df_final, is_training=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'📊 Registros de Entrenamiento: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'📐 Variables de entrada procesadas: {X_train.shape[1]}')
print('🎯 Distribución de Clases (0: Bajo, 1: Medio, 2: Alto):')
print(pd.Series(y).value_counts().sort_index())

In [ ]:
# Cálculo explícito de pesos por desbalance de clases
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print('⚖️ Pesos asignados por clase para equilibrar la función de pérdida:')
for k, v in class_weight_dict.items():
    print(f'   - Clase {k}: {v:.4f}')

In [ ]:
print('🧠 Entrenando Perceptrón Multicapa (MLP) con Keras...')
mlp = MLPClassifier(input_dim=X_train.shape[1], num_classes=3)
history = mlp.entrenar(X_train, y_train, epochs=40, batch_size=32)

y_pred_mlp = mlp.predecir(X_test)
print('✅ Entrenamiento del MLP completado.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Curva de Pérdida
ax1.plot(history.history['loss'], label='Pérdida Entrenamiento', linewidth=2)
ax1.plot(history.history['val_loss'], label='Pérdida Validación', linestyle='--', linewidth=2)
ax1.set_title('Curva de Pérdida (Loss) - MLP', fontsize=12, fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('Categorical Cross-Entropy')
ax1.legend()

# Curva de Exactitud
ax2.plot(history.history['accuracy'], label='Accuracy Entrenamiento', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Accuracy Validación', linestyle='--', linewidth=2)
ax2.set_title('Curva de Exactitud (Accuracy) - MLP', fontsize=12, fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
print('📊 Entrenando y evaluando 5 Modelos de Línea Base (scikit-learn)...')
evaluador = BaselineEvaluator()
df_baseline = evaluador.entrenar_y_evaluar_todos(X_train, y_train, X_test, y_test)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp, average='macro', zero_division=0)
rec_mlp = recall_score(y_test, y_pred_mlp, average='macro', zero_division=0)
f1_mlp = f1_score(y_test, y_pred_mlp, average='macro', zero_division=0)

fila_mlp = pd.DataFrame([{
    'Modelo': 'MLP (Propio - Keras)',
    'Accuracy': round(acc_mlp, 4),
    'Precision (Macro)': round(prec_mlp, 4),
    'Recall (Macro)': round(rec_mlp, 4),
    'F1-Score (Macro)': round(f1_mlp, 4)
}])

tabla_comparativa = pd.concat([fila_mlp, df_baseline], ignore_index=True)

print('\n' + '='*65)
print('🏆 TABLA COMPARATIVA CONSOLIDADA (6 MODELOS EVALUADOS)')
print('='*65)
display(tabla_comparativa)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

todos_los_modelos = {'MLP (Keras)': y_pred_mlp}
for nombre, modelo in evaluador.modelos_entrenados.items():
    todos_los_modelos[nombre] = modelo.predict(X_test)

clases_labels = ['Bajo (0)', 'Medio (1)', 'Alto (2)']

for idx, (nombre, preds) in enumerate(todos_los_modelos.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=clases_labels, yticklabels=clases_labels)
    axes[idx].set_title(f'Matriz de Confusión: {nombre}', fontweight='bold')
    axes[idx].set_xlabel('Predicción')
    axes[idx].set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
print('🔍 Calculando valores de Shapley con SHAP para la red MLP...')
explainer = SHAPExplainer(mlp.model.predict, X_train, preprocesador.feature_names)
fig_shap = explainer.generar_grafico_resumen(X_test, n_samples=20)
plt.show()